# Gold Dimension: TfL Line

Build the historical Tube-line dimension using Slowly Changing Dimension Type 2.

The dimension preserves previous versions when descriptive line attributes change.

This notebook:

1. Reads Silver line-status data.
2. Selects the latest complete TfL snapshot.
3. Creates an attribute hash for change detection.
4. Validates source uniqueness.
5. Identifies new, changed, and unchanged lines.
6. Expires changed dimension versions.
7. Inserts new dimension versions.
8. Validates SCD Type 2 integrity.

**Source:** `workspace.urbanpulse_silver.tfl_line_status`

**Target:** `workspace.urbanpulse_gold.dim_line`

**Business key:** `line_id`

**Grain:** One row per historical version of a Tube line.

## 1. Initialise project paths

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

## 2. Import dependencies

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

from urbanpulse.transformations.dim_line import (
    prepare_line_source,
)

## 3. Define source and target tables

In [0]:
SILVER_TABLE = (
    "workspace."
    "urbanpulse_silver."
    "tfl_line_status"
)

TARGET_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "dim_line"
)

## 4. Prepare the latest TfL line snapshot

Line reference attributes are taken from the latest available Silver snapshot.

Multiple status records for a line are reduced to one reference record because operational status is not part of the line dimension.

In [0]:
silver_df = spark.table(
    SILVER_TABLE
)

source_df = prepare_line_source(
    silver_df
)

print(
    f"Latest source lines: "
    f"{source_df.count()}"
)

display(
    source_df
    .orderBy("line_name")
)

## 5. Validate source business keys

The latest source snapshot must contain exactly one record per `line_id`.

In [0]:
if source_df.count() == 0:
    raise ValueError(
        "Latest line snapshot contains no lines."
    )


null_ids = (
    source_df
    .filter(
        F.col("line_id").isNull()
    )
    .count()
)


duplicate_ids_df = (
    source_df
    .groupBy("line_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)


duplicate_ids = (
    duplicate_ids_df.count()
)


if null_ids > 0:
    raise ValueError(
        "Null line IDs detected."
    )


if duplicate_ids > 0:
    display(duplicate_ids_df)

    raise ValueError(
        "Duplicate line IDs detected "
        "in the latest source snapshot."
    )


print(
    "Line source validation passed."
)

## 6. Detect initial or incremental processing

The first execution creates the dimension.

Later executions compare the latest source state with current dimension records.

In [0]:
TARGET_EXISTS = (
    spark.catalog.tableExists(
        TARGET_TABLE
    )
)

print(
    f"Target exists: "
    f"{TARGET_EXISTS}"
)

## 7. Create the initial SCD Type 2 versions

For the first load:

- `effective_from` is the source snapshot timestamp
- `effective_to` is null
- `is_current` is true
- every source line becomes the first dimension version

In [0]:
if not TARGET_EXISTS:

    initial_df = (
        source_df
        .withColumn(
            "effective_from",
            F.col("snapshot_at"),
        )
        .withColumn(
            "effective_to",
            F.lit(None).cast("timestamp"),
        )
        .withColumn(
            "is_current",
            F.lit(True),
        )
        .withColumn(
            "line_key",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.col("line_id"),
                    F.col("attribute_hash"),
                    F.col(
                        "effective_from"
                    ).cast("string"),
                ),
                256,
            ),
        )
        .withColumn(
            "created_at",
            F.current_timestamp(),
        )
        .withColumn(
            "updated_at",
            F.current_timestamp(),
        )
        .select(
            "line_key",
            "line_id",
            "line_name",
            "mode_name",
            "is_active",
            "attribute_hash",
            "effective_from",
            "effective_to",
            "is_current",
            "created_at",
            "updated_at",
        )
    )

    (
        initial_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            TARGET_TABLE
        )
    )

    print(
        f"Initial dimension created: "
        f"{TARGET_TABLE}"
    )

## 8. Load current dimension versions

Only rows where `is_current = true` participate in change detection.

In [0]:
target_df = spark.table(
    TARGET_TABLE
)

current_df = (
    target_df
    .filter(
        F.col("is_current")
    )
)

print(
    f"Current dimension lines: "
    f"{current_df.count()}"
)

## 9. Validate source coverage

A line that exists as current in Gold but is absent from the latest source snapshot is treated as an unexpected source condition.

The pipeline stops rather than automatically interpreting absence as deactivation.

In [0]:
missing_from_source_df = (
    current_df
    .select(
        "line_id",
        "line_name",
    )
    .join(
        source_df.select("line_id"),
        on="line_id",
        how="left_anti",
    )
)

missing_count = (
    missing_from_source_df.count()
)

if missing_count > 0:
    display(
        missing_from_source_df
    )

    raise ValueError(
        f"{missing_count} current Gold lines "
        "are missing from the latest source snapshot."
    )

print(
    "Source coverage validation passed."
)

## 10. Identify new lines

A new line exists in the latest source but has no current Gold dimension record.

In [0]:
new_lines_df = (
    source_df.alias("source")
    .join(
        current_df
        .select("line_id")
        .alias("target"),
        on="line_id",
        how="left_anti",
    )
)

new_count = (
    new_lines_df.count()
)

print(
    f"New lines: {new_count}"
)

## 11. Identify changed lines

A line is considered changed when its business key still exists but its descriptive attribute hash differs from the current Gold version.

In [0]:
changed_lines_df = (
    source_df.alias("source")
    .join(
        current_df.alias("target"),
        on="line_id",
        how="inner",
    )
    .filter(
        F.col("source.attribute_hash")
        !=
        F.col("target.attribute_hash")
    )
    .select(
        "source.*"
    )
)

changed_count = (
    changed_lines_df.count()
)

print(
    f"Changed lines: {changed_count}"
)

## 12. Review unchanged lines

In [0]:
unchanged_lines_df = (
    source_df.alias("source")
    .join(
        current_df.alias("target"),
        on="line_id",
        how="inner",
    )
    .filter(
        F.col("source.attribute_hash")
        ==
        F.col("target.attribute_hash")
    )
    .select(
        "source.line_id",
        "source.line_name",
    )
)

unchanged_count = (
    unchanged_lines_df.count()
)

print(
    f"Unchanged lines: "
    f"{unchanged_count}"
)

## 13. Review SCD change detection

In [0]:
print(
    "SCD Type 2 change summary"
)

print(
    f"New:       {new_count}"
)

print(
    f"Changed:   {changed_count}"
)

print(
    f"Unchanged: {unchanged_count}"
)

## 14. Expire changed dimension versions

Changed current rows are closed using the new source snapshot timestamp.

The validity interval is:

`effective_from <= timestamp < effective_to`

In [0]:
if changed_count > 0:

    changed_keys_df = (
        changed_lines_df
        .select(
            "line_id",
            F.col(
                "snapshot_at"
            ).alias(
                "change_timestamp"
            ),
        )
    )

    target_delta = (
        DeltaTable.forName(
            spark,
            TARGET_TABLE,
        )
    )

    (
        target_delta.alias("target")
        .merge(
            changed_keys_df.alias("source"),
            """
            target.line_id = source.line_id
            AND target.is_current = true
            """,
        )
        .whenMatchedUpdate(
            set={
                "effective_to":
                    "source.change_timestamp",

                "is_current":
                    "false",

                "updated_at":
                    "current_timestamp()",
            }
        )
        .execute()
    )

    print(
        f"Expired {changed_count} "
        "dimension versions."
    )

else:
    print(
        "No dimension versions "
        "require expiration."
    )

## 15. Prepare new dimension versions

New source lines and changed source lines receive new current SCD Type 2 versions.

In [0]:
versions_to_insert_df = (
    new_lines_df
    .unionByName(
        changed_lines_df
    )
)

In [0]:
versions_to_insert_df = (
    versions_to_insert_df
    .withColumn(
        "effective_from",
        F.col("snapshot_at"),
    )
    .withColumn(
        "effective_to",
        F.lit(None).cast("timestamp"),
    )
    .withColumn(
        "is_current",
        F.lit(True),
    )
    .withColumn(
        "line_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("line_id"),
                F.col("attribute_hash"),
                F.col(
                    "effective_from"
                ).cast("string"),
            ),
            256,
        ),
    )
    .withColumn(
        "created_at",
        F.current_timestamp(),
    )
    .withColumn(
        "updated_at",
        F.current_timestamp(),
    )
    .select(
        "line_key",
        "line_id",
        "line_name",
        "mode_name",
        "is_active",
        "attribute_hash",
        "effective_from",
        "effective_to",
        "is_current",
        "created_at",
        "updated_at",
    )
)

## 16. Insert new SCD versions

Only new or changed lines are appended.

Unchanged records are not rewritten.

In [0]:
insert_count = (
    versions_to_insert_df.count()
)

if insert_count > 0:

    (
        versions_to_insert_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(
            TARGET_TABLE
        )
    )

    print(
        f"Inserted {insert_count} "
        "new dimension versions."
    )

else:
    print(
        "No new dimension versions "
        "to insert."
    )

## 17. Verify current dimension records

In [0]:
%sql
SELECT
    line_key,
    line_id,
    line_name,
    mode_name,
    is_active,
    effective_from,
    effective_to,
    is_current
FROM workspace.urbanpulse_gold.dim_line
WHERE is_current = TRUE
ORDER BY line_name;

## 18. Verify one current version per business key

In [0]:
%sql
SELECT
    line_id,
    COUNT(*) AS current_versions
FROM workspace.urbanpulse_gold.dim_line
WHERE is_current = TRUE
GROUP BY line_id
HAVING COUNT(*) <> 1;

In [0]:
%sql
SELECT
    line_key,
    COUNT(*) AS records
FROM workspace.urbanpulse_gold.dim_line
GROUP BY line_key
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT *
FROM workspace.urbanpulse_gold.dim_line
WHERE
    effective_to IS NOT NULL
    AND effective_to <= effective_from;

In [0]:
%sql
SELECT *
FROM workspace.urbanpulse_gold.dim_line
WHERE
    is_current = TRUE
    AND effective_to IS NOT NULL;

In [0]:
%sql
SELECT *
FROM workspace.urbanpulse_gold.dim_line
WHERE
    is_current = FALSE
    AND effective_to IS NULL;

## 19. Review dimensional history

A line normally has one row until a tracked attribute changes.

When a change occurs, the previous version remains available for historical fact joins.

In [0]:
%sql
SELECT
    line_id,
    line_name,
    mode_name,
    effective_from,
    effective_to,
    is_current
FROM workspace.urbanpulse_gold.dim_line
ORDER BY
    line_id,
    effective_from;

In [0]:
%sql
SELECT COUNT(*) AS versions
FROM workspace.urbanpulse_gold.dim_line;